# Görüntü Oluşturma Uygulamaları Geliştirme (Azure OpenAI)

LLM'ler sadece metin üretmekle kalmaz. Metin açıklamalarından görüntü de oluşturabilirsiniz. Görüntüler bir modalite olarak MedTech, mimari, turizm, oyun geliştirme, pazarlama ve daha birçok alanda faydalıdır. Bu derste, Microsoft Foundry üzerindeki günümüz **GPT Görüntü** modelleri ile çalışıyoruz.

## Öğrenme hedefleri

Bu dersin sonunda şunları yapabileceksiniz:

- Görüntü oluşturmanın ne olduğunu ve nerelerde faydalı olduğunu açıklamak.
- `gpt-image` model ailesini ve bunun eski DALL·E modellerinden nasıl farklı olduğunu anlamak.
- Bir görüntü oluşturma uygulaması geliştirmek ve görüntüleri düzenlemek.

## Görüntü oluşturma nedir?

Görüntü oluşturma modelleri metin isteminden görüntüler yaratır. `gpt-image` gibi modern modeller, eğitim sırasında metin ile görüntü arasındaki ilişkiyi öğrenir, sonra rastgele gürültüyü, isteminize uygun bir görüntüye dönüştürür.

İki iyi bilinen görüntü modeli ailesi şunlardır:

- **`gpt-image` (OpenAI)** — bu derste kullanılan güncel nesil. Metinden görüntü oluşturmayı ve görüntü düzenlemeyi (maske ile boyama) destekler.
- **Midjourney** — kendi servisi ve Discord tabanlı iş akışı olan popüler üçüncü taraf model.

> Eski OpenAI görüntü modelleri — **DALL·E 2** ve **DALL·E 3** — eski nesildir. DALL·E 3 artık yeni dağıtımlar için mevcut değil, `create_variation` gibi özellikler sadece DALL·E 2’de vardı. Yeni uygulamalar için `gpt-image` modellerini kullanın.

Microsoft Foundry’de, **`gpt-image-2`** en yeni ve en yetenekli görüntü modeli olup varsayılan olarak önerilir. `gpt-image-1.5` ve `gpt-image-1-mini` de genel olarak kullanılabilir durumdadır.

> **Önemli:** `gpt-image` modelleri oluşturulan görüntüyü URL olarak değil, **base64** (`b64_json`) formatında döndürür. Kodunuz base64 dizisini baytlara çözer ve kaydeder — indirilecek bir görüntü URL'si yoktur.


## İlk görüntü oluşturma uygulamanızı geliştirmek

Peki, bir görüntü oluşturma uygulaması geliştirmek için ne gerekiyor? Şu kütüphanelere ihtiyacınız var:

- **python-dotenv**, gizli bilgilerinizi koddan uzak tutmak için bu kütüphaneyi *.env* dosyasında tutmanız şiddetle önerilir.
- **openai**, OpenAI API ile etkileşimde bulunmak için kullanacağınız kütüphane.
- **pillow**, Python'da görüntülerle çalışmak için.

Eğer henüz yapmadıysanız, [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) sayfasındaki talimatları takip ederek bir Microsoft Foundry kaynağı ve modeli oluşturun. Model olarak **gpt-image-2**'yi seçin (en son Azure OpenAI görüntü modeli; DALL·E eski).

1. Aşağıdaki içeriğe sahip *.env* adlı bir dosya oluşturun:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Bu bilgileri kaynak için Microsoft Foundry portalında "Deployments" bölümünde bulun.



1. Yukarıdaki kütüphaneleri *requirements.txt* adlı bir dosyada aşağıdaki gibi toplayın:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ardından, sanal ortam oluşturun ve kütüphaneleri yükleyin:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windows için, sanal ortamınızı oluşturmak ve etkinleştirmek için aşağıdaki komutları kullanın:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* adlı dosyaya aşağıdaki kodu ekleyin:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # dotenv'i içe aktar
    dotenv.load_dotenv()

    # Azure OpenAI (Microsoft Foundry) istemcisini yapılandır.
    # Görüntü modelleri güncel bir API sürümü gerektirir — modelinizin ihtiyaç duyduğu sürümü Foundry belgelerinden kontrol edin.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Görüntü oluşturma API'sini kullanarak bir görüntü oluştur
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Buraya istem metninizi girin
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # örn. "gpt-image-2"
        )
        # Saklanan görüntü için dizini ayarla
        image_dir = os.path.join(os.curdir, 'images')

        # Dizin yoksa, oluştur
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Görüntü yolunu başlat (dosya türünün png olması gerektiğine dikkat et)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modelleri görüntüyü URL olarak değil base64 (b64_json) olarak döndürür
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Görüntüyü varsayılan görüntüleyicide göster
        image = Image.open(image_path)
        image.show()

    # istisnaları yakala
    except BadRequestError as err:
        print(err)

    ```

Bu kodu açıklayalım:

- İlk olarak, OpenAI kütüphanesi, dotenv kütüphanesi, base64 modülü ve Pillow kütüphanesi dahil olmak üzere ihtiyacımız olan kütüphaneleri içe aktarıyoruz.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Sonra, *.env* dosyasından ortam değişkenlerini yüklüyoruz.

    ```python
    # dotenv ithalat et
    dotenv.load_dotenv()
    ```

- Ardından, Azure OpenAI (Microsoft Foundry) istemcisini yapılandırıyoruz.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Sonra, resmi oluşturuyoruz:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # İsteminizi buraya girin
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` modelleri görüntüyü `data[0].b64_json` içinde **base64** dizgesi olarak döndürür. Bunun decode edilip байtlara çevrilmesi ve dosyaya yazılması gerekir — indirilecek bir URL yoktur.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Son olarak, resmi açar ve standart resim görüntüleyici ile gösteririz:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Resim oluşturma hakkında daha fazla detay

`images.generate` parametrelerine bakalım:

- **prompt**, resmi oluşturmak için kullanılan metin komutu. Burada "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils" (Bir at üstünde, lolipop tutan tavşan, sümbüllerin yetiştiği sisli bir çayırda).
- **size**, oluşturulan resmin boyutudur (örneğin `1024x1024`, `1536x1024`, `1024x1536` veya `"auto"`).
- **n**, oluşturulan resim sayısıdır. Burada bir tane oluşturuyoruz.
- **model**, resim model dağıtımınızın adıdır (örneğin `gpt-image-2`).

> Resim modelleri `temperature` parametresi almaz — bu bir metin üretim kontrolüdür. Çeşitlilik için API'yi tekrar çağırın; çeşitliliği azaltmak için komutunuzu daha spesifik yapın.

## Resim oluşturmanın ek yetenekleri

Python'da birkaç satır kodla nasıl resim oluşturulacağını gördünüz. `gpt-image` modelleri aynı zamanda mevcut bir resmi **düzenleyebilir**. Bir resim, isteğe bağlı **maske** (değiştirilecek alanı işaretler) ve komut sağlayarak, resmin bir bölümünü değiştirebilirsiniz — örneğin, tavşanımıza bir şapka eklemek gibi.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# düzenlemeler de base64 olarak döndürülür
edited_image = base64.b64decode(response.data[0].b64_json)
```

Temel resimde sadece tavşan var; son resimde tavşanın üstünde şapka bulunuyor.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
